In [ ]:
!pip install transformers scikit-learn torch -q

In [ ]:
import pandas as pd
import numpy as np
import torch

print("CUDA Available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

In [ ]:
train_df = pd.read_csv('/content/twitter_training.csv', header=None)
val_df = pd.read_csv('/content/twitter_validation.csv', header=None)

In [ ]:
train_df.columns = ['id','entity','label','text']
val_df.columns = ['id','entity','label','text']

In [ ]:
train_df.dropna(inplace=True)
val_df.dropna(inplace=True)

train_df = train_df[['text','label']]
val_df = val_df[['text','label']]

# Reduce dataset size (FAST)
train_df = train_df.sample(5000, random_state=42)
val_df = val_df.sample(1000, random_state=42)

# Label encoding
label_map = {
    'Negative': 0,
    'Neutral': 1,
    'Positive': 2,
    'Irrelevant': 3
}

train_df['label'] = train_df['label'].map(label_map)
val_df['label'] = val_df['label'].map(label_map)

In [ ]:
train_texts, test_texts, train_labels, test_labels = train_test_split(
    train_df['text'], train_df['label'], test_size=0.2, random_state=42
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

def tokenize(texts):
    return tokenizer(list(texts), padding=True, truncation=True, max_length=128)

train_encodings = tokenize(train_texts)
val_encodings = tokenize(val_df['text'])
test_encodings = tokenize(test_texts)

In [ ]:
class TwitterDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = TwitterDataset(train_encodings, train_labels)
val_dataset = TwitterDataset(val_encodings, val_df['label'])
test_dataset = TwitterDataset(test_encodings, test_labels)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=4
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

In [ ]:
training_args = TrainingArguments(
    output_dir='./results',
    learning_rate=2e-5,
    per_device_train_batch_size=32,   # 🔥 faster
    per_device_eval_batch_size=32,
    num_train_epochs=1,
    logging_steps=20
)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')
    acc = accuracy_score(labels, preds)

    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

In [ ]:
predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=1)

print("Accuracy:", accuracy_score(test_labels, preds))

precision, recall, f1, _ = precision_recall_fscore_support(test_labels, preds, average='weighted')

print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

print("Confusion Matrix:\n", confusion_matrix(test_labels, preds))

Analysis
- Used DistilBERT for faster training
- Reduced dataset size for efficiency
- Achieved good performance with minimal training
- Handled noisy Twitter data
- GPU significantly improved speed